In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Localiza colunas de Idade e Doenças Crônicas
col_idade = 'Idade'
cols_cronicas = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c]

# ==============================================================================
# 2. PREPARAÇÃO DOS DADOS (LONGITUDINAL -> ÚNICOS)
# ==============================================================================
df['Idade_Num'] = pd.to_numeric(df[col_idade], errors='coerce')

df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
colunas_propagacao = ['Idade_Num'] + cols_cronicas
colunas_existentes = [c for c in colunas_propagacao if c in df.columns]

df[colunas_existentes] = df.groupby('Record ID')[colunas_existentes].ffill()
df_unicos = df.drop_duplicates(subset=['Record ID'], keep='last').copy()

# ==============================================================================
# 3. RESTRUTURAÇÃO (MELT) E LIMPEZA AVANÇADA
# ==============================================================================
df_melted = df_unicos.melt(id_vars=['Record ID', 'Idade_Num'],
                           value_vars=cols_cronicas,
                           var_name='Doenca',
                           value_name='Status')

# Fica apenas com quem tem a doença marcada
df_doentes = df_melted[df_melted['Status'] == 'Checked'].copy()

# Limpa o texto da doença
df_doentes['Doenca_Nome'] = df_doentes['Doenca'].apply(lambda x: str(x).split('choice=')[-1].replace(')', '').strip())

# ▼ REMOÇÃO DO "NÃO SE APLICA" E OUTROS RUÍDOS ▼
termos_inuteis = ['não se aplica', 'nao se aplica', 'nenhuma', 'nenhum', 'outras', 'outro']
padrao_exclusao = '|'.join(termos_inuteis)
df_doentes = df_doentes[~df_doentes['Doenca_Nome'].str.contains(padrao_exclusao, case=False, na=False)]

# Remove quem está sem idade preenchida
df_doentes = df_doentes.dropna(subset=['Idade_Num'])

if df_doentes.empty:
    print("Nenhuma doença crônica válida (com idade) foi reportada nesta base.")
else:
    # Pegamos as Top 8 doenças para um visual equilibrado (nem vazio, nem poluído)
    top_doencas = df_doentes['Doenca_Nome'].value_counts().nlargest(8).index
    df_plot = df_doentes[df_doentes['Doenca_Nome'].isin(top_doencas)].copy()

    # ==============================================================================
    # 4. GERAÇÃO DO RIDGELINE PLOT ESTÉTICO
    # ==============================================================================
    fig = go.Figure()

    # Paleta de cores mais sofisticada e executiva
    colors = px.colors.qualitative.Bold

    # Invertemos a ordem para a doença #1 ficar no topo
    for i, doenca in enumerate(reversed(top_doencas)):
        idade_array = df_plot[df_plot['Doenca_Nome'] == doenca]['Idade_Num']
        cor_atual = colors[i % len(colors)]

        # Encurta nomes excessivamente longos
        nome_formatado = str(doenca)[:35] + '...' if len(str(doenca)) > 35 else str(doenca)

        fig.add_trace(go.Violin(
            x=idade_array,
            name=nome_formatado,
            side='positive',
            width=2.5,            # Sobreposição otimizada para não virar uma "nuvem" de cor
            points=False,
            line_color='black',   # CONTORNO ESCURO: Define a separação das montanhas
            line_width=1.2,       # Espessura da linha do contorno
            fillcolor=cor_atual,
            opacity=0.85,
            meanline_visible=True # ADICIONA A LINHA DA MÉDIA dentro de cada montanha!
        ))

    # ==============================================================================
    # 5. AJUSTES FINOS DE LAYOUT (TELA DE PINTURA LIMPA)
    # ==============================================================================
    fig.update_layout(
        title=dict(
            text="Evolução da Idade por Doença Crônica (Ridgeline Plot)",
            font=dict(size=20, family="Arial", color='#2c3e50'),
            x=0.05
        ),
        xaxis=dict(
            title="Idade do Paciente (Anos)",
            showgrid=True,
            gridcolor='#f0f0f0', # Grade X muito sutil
            zeroline=False,
            title_font=dict(size=14)
        ),
        yaxis=dict(
            showgrid=False, # REMOVE A GRADE Y para não cortar as montanhas
            zeroline=False
        ),
        height=700,
        margin=dict(l=280, r=50, t=100, b=80),
        showlegend=False,
        plot_bgcolor='white',  # Fundo perfeitamente branco
        paper_bgcolor='white'
    )

    fig.show()

    # ==============================================================================
    # 6. GERAÇÃO E EXPORTAÇÃO DA TABELA ESTATÍSTICA (BASES DAS MONTANHAS)
    # ==============================================================================
    # Agrupa por doença e calcula a contagem e os quartis/médias de idade
    df_tabela = df_plot.groupby('Doenca_Nome')['Idade_Num'].agg(
        Quantidade_Pacientes='count',
        Idade_Minima='min',
        Idade_Media='mean',
        Idade_Mediana='median',
        Idade_Maxima='max'
    ).reset_index()

    # Arredondamos as idades médias para 1 casa decimal para ficar limpo no Excel
    df_tabela['Idade_Media'] = df_tabela['Idade_Media'].round(1)

    # Ordenamos pelo volume de pacientes (do maior para o menor, acompanhando o gráfico)
    df_tabela = df_tabela.sort_values(by='Quantidade_Pacientes', ascending=False).reset_index(drop=True)

    # Imprimimos a tabela no console de forma elegante
    print("\n" + "="*95)
    print("TABELA ESTATÍSTICA: DISTRIBUIÇÃO DE IDADE POR DOENÇA CRÔNICA (Top 8)")
    print("="*95)
    print(df_tabela.to_string(index=False))
    print("="*95)

    # Exportamos silenciosamente para o seu computador
    nome_arquivo = 'Tabela_Ridgeline_Idades_DoencasCronicas.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
    print(f"\n[SUCESSO] A tabela base do gráfico foi salva na sua máquina como '{nome_arquivo}'.\n")

In [ ]:
# ==============================================================================
# ENVELHECIMENTO E SAÚDE: CARGA DE DOENÇAS CRÔNICAS A PARTIR DOS 45 ANOS
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARREGAMENTO DOS DADOS
# ------------------------------------------------------------------------------
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# ------------------------------------------------------------------------------
# 2. PREPARAÇÃO DA IDADE E FILTRO DE ATENDIMENTOS
# ------------------------------------------------------------------------------
col_idade = 'Idade'
df['Idade_Num'] = pd.to_numeric(df[col_idade], errors='coerce')

# Propagamos a Idade do cadastro para todas as consultas do paciente
df = df.sort_values(['Record ID', 'Repeat Instance'], na_position='first')
df['Idade_Num'] = df.groupby('Record ID')['Idade_Num'].ffill()

# Filtramos apenas as consultas (eventos de atendimento)
df_ev = df[df['Repeat Instrument'].notna()].copy()

# Removemos consultas onde a idade não pôde ser identificada (para bater o N=29142)
df_ev = df_ev.dropna(subset=['Idade_Num'])

# Criação da Faixa Etária
df_ev['Grupo_Etario'] = np.where(df_ev['Idade_Num'] >= 45, '45 anos ou mais', 'Menos de 45 anos')

# ------------------------------------------------------------------------------
# 3. RASTREAMENTO CLÍNICO (TEXT MINING + CHECKBOXES)
# ------------------------------------------------------------------------------
text_cols = [c for c in df_ev.columns if df_ev[c].dtype == object]
df_text = df_ev[text_cols].fillna('')

# HAS (Hipertensão)
regex_has = r'\b(hipertens[aã]o|has|press[aã]o alta)\b'
cond_has_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_has, case=False, regex=True)).any(axis=1)

# DM (Diabetes)
regex_dm = r'\b(diabetes|dm2|dm1|dm)\b'
cond_dm_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_dm, case=False, regex=True)).any(axis=1)

# Checkboxes estruturados
cols_cronicas = [c for c in df_ev.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower()]
cond_has_str = pd.Series(False, index=df_ev.index)
cond_dm_str = pd.Series(False, index=df_ev.index)

for c in cols_cronicas:
    is_checked = df_ev[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
    if 'hipertensão' in c.lower() or 'has' in c.lower():
        cond_has_str = cond_has_str | is_checked
    if 'diabetes' in c.lower() or 'dm' in c.lower():
        cond_dm_str = cond_dm_str | is_checked

df_ev['Tem_HAS'] = (cond_has_text | cond_has_str).astype(int)
df_ev['Tem_DM'] = (cond_dm_text | cond_dm_str).astype(int)

# ------------------------------------------------------------------------------
# 4. CÁLCULO ESTATÍSTICO E CRIAÇÃO DA TABELA
# ------------------------------------------------------------------------------
# Qui-Quadrado para HAS e DM
tabela_has = pd.crosstab(df_ev['Grupo_Etario'], df_ev['Tem_HAS'])
p_has = chi2_contingency(tabela_has, correction=False)[1]

tabela_dm = pd.crosstab(df_ev['Grupo_Etario'], df_ev['Tem_DM'])
p_dm = chi2_contingency(tabela_dm, correction=False)[1]

resultados = []
grupos = ['Menos de 45 anos', '45 anos ou mais']

for grupo in grupos:
    df_grupo = df_ev[df_ev['Grupo_Etario'] == grupo]
    total_atend = len(df_grupo)
    
    if total_atend > 0:
        has_abs = df_grupo['Tem_HAS'].sum()
        dm_abs = df_grupo['Tem_DM'].sum()
        
        resultados.append({
            'Diagnóstico': 'Hipertensão',
            'Grupo Etário': grupo,
            'Total_Atendimentos': total_atend,
            'Casos': has_abs,
            'Prevalência (%)': round((has_abs / total_atend) * 100, 2),
            'P_Value': p_has
        })
        
        resultados.append({
            'Diagnóstico': 'Diabetes',
            'Grupo Etário': grupo,
            'Total_Atendimentos': total_atend,
            'Casos': dm_abs,
            'Prevalência (%)': round((dm_abs / total_atend) * 100, 2),
            'P_Value': p_dm
        })

df_tabela = pd.DataFrame(resultados)

# ==============================================================================
# 5. EXIBIÇÃO DO RELATÓRIO EXECUTIVO (CONFORME IMAGEM)
# ==============================================================================
total_geral = len(df_ev)
total_menos45 = len(df_ev[df_ev['Grupo_Etario'] == 'Menos de 45 anos'])
total_mais45 = len(df_ev[df_ev['Grupo_Etario'] == '45 anos ou mais'])

has_menos45 = df_tabela[(df_tabela['Diagnóstico'] == 'Hipertensão') & (df_tabela['Grupo Etário'] == 'Menos de 45 anos')]['Prevalência (%)'].values[0]
has_mais45 = df_tabela[(df_tabela['Diagnóstico'] == 'Hipertensão') & (df_tabela['Grupo Etário'] == '45 anos ou mais')]['Prevalência (%)'].values[0]

dm_menos45 = df_tabela[(df_tabela['Diagnóstico'] == 'Diabetes') & (df_tabela['Grupo Etário'] == 'Menos de 45 anos')]['Prevalência (%)'].values[0]
dm_mais45 = df_tabela[(df_tabela['Diagnóstico'] == 'Diabetes') & (df_tabela['Grupo Etário'] == '45 anos ou mais')]['Prevalência (%)'].values[0]

print("\n" + "="*85)
print("SÍNTESE ESTATÍSTICA: CARGA DE DOENÇAS CRÔNICAS E ENVELHECIMENTO")
print("="*85)
print(f"Total de Atendimentos Analisados (com Idade válida): {total_geral}")
print(f" -> Menos de 45 anos:  {total_menos45}")
print(f" -> 45 anos ou mais:   {total_mais45}\n")

print("[1] HIPERTENSÃO ARTERIAL (HAS)")
print(f"    - Prevalência < 45 anos: {has_menos45:.2f}%")
print(f"    - Prevalência ≥ 45 anos: {has_mais45:.2f}%")
print(f"    - Aumento Estatístico:   p-value = {p_has:.4e} (Significativo)\n")

print("[2] DIABETES MELLITUS (DM)")
print(f"    - Prevalência < 45 anos: {dm_menos45:.2f}%")
print(f"    - Prevalência ≥ 45 anos: {dm_mais45:.2f}%")
print(f"    - Aumento Estatístico:   p-value = {p_dm:.4e} (Significativo)")
print("="*85 + "\n")

# Exportação Silenciosa da Tabela
nome_arquivo = 'Tabela_Carga_Cronicas_Idade.csv'
df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
print(f"📂 Tabela base gerada silenciosamente em: '{nome_arquivo}'.\n")

# ==============================================================================
# 6. GERAÇÃO DO GRÁFICO (DESIGN SEABORN IDÊNTICO)
# ==============================================================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Paleta exata: Azul (#4a90e2) e Terracota (#d55e52)
cores = ['#4a90e2', '#d55e52']

ax = sns.barplot(
    data=df_tabela,
    x='Diagnóstico',
    y='Prevalência (%)',
    hue='Grupo Etário',
    palette=cores,
    edgecolor='white',
    linewidth=1.5
)

plt.title('Evolução da Carga de Doenças Crônicas a partir dos 45 anos', 
          fontsize=14, pad=20, color='#333333')
plt.ylabel('Prevalência na População Atendida (%)', fontsize=12, color='#333333')
plt.xlabel('Diagnóstico', fontsize=12, color='#333333')

# Aumenta o teto para os rótulos não cortarem
plt.ylim(0, max(df_tabela['Prevalência (%)']) * 1.3)

# Rótulos precisos em cima das barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=4, fontweight='bold', color='#2c3e50', fontsize=11)

# Estética limpa (remoção de bordas superiores e direitas)
sns.despine(left=False, bottom=False, top=True, right=True)

# Ajuste da legenda
plt.legend(title='Grupo Etário', title_fontsize='11', fontsize='10', loc='upper right', frameon=True)

plt.tight_layout()
plt.show()